# PUEEA
> **Contexto de dominio:** `docs/fuentes/pueea.md`  
> **Bloque:** A  
> **Variable principal:** consumo_agua (m³)  
> Ejecutar `Plan.md` sección 3 (ciclo estadístico completo).

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models

print("Setup OK | Modelos disponibles:", list_models())

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Documentar la fuente en `docs/fuentes/pueea.md`.

In [ ]:
# df = load_csv("data/raw/pueea.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":      pd.date_range("2015-01-01", periods=n, freq="ME"),
    "consumo_agua": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
df.head()

## 2. Validación y EDA

In [ ]:
val = validate(df, date_col="fecha")
print(val.summary())
cat = classify(df)
print(cat.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_{fuente_key}.html",
       title="EDA — PUEEA", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "consumo_agua", title="PUEEA")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "consumo_agua", period="month")
plt.show()

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial

In [ ]:
ts = df.set_index("fecha")["consumo_agua"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["consumo_agua"], method="linear")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["consumo_agua"]

models = {
    "SARIMA":      get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "XGBoost":     get_model("xgboost", lags=[1,2,3,6,12]),
    "RandomForest":get_model("random_forest", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    results[name] = walk_forward(model, ts, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

- Variable analizada: **consumo_agua** (m³)
- Fuente de dominio: `docs/fuentes/pueea.md`
- Completar con hallazgos reales al trabajar con datos de producción.

### Referencias
- Ver `docs/fuentes/pueea.md` para normativa, indicadores y fuentes de datos.